<a href="https://colab.research.google.com/github/DanielHashmi/Homework_Python_Projects/blob/main/25%20Projects/Space_Invaders_Game.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pygame
import random
import sys
import math
from pygame import mixer

pygame.init()

WIDTH, HEIGHT = 800, 600
screen = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Space Invaders")

clock = pygame.time.Clock()
FPS = 60

BLACK = (0, 0, 0)
WHITE = (255, 255, 255)
GREEN = (0, 255, 0)
RED = (255, 0, 0)
BLUE = (0, 0, 255)
YELLOW = (255, 255, 0)
PURPLE = (128, 0, 128)

font = pygame.font.SysFont('arial', 32)
small_font = pygame.font.SysFont('arial', 16)

class Player:
    def __init__(self):
        self.width = 60
        self.height = 40
        self.x = WIDTH // 2 - self.width // 2
        self.y = HEIGHT - self.height - 20
        self.speed = 7
        self.bullets = []
        self.cooldown = 0
        self.cooldown_duration = 10
        self.lives = 3

    def draw(self):
        points = [
            (self.x + self.width // 2, self.y),
            (self.x, self.y + self.height),
            (self.x + self.width, self.y + self.height)
        ]
        pygame.draw.polygon(screen, GREEN, points)

        pygame.draw.rect(screen, GREEN, (self.x + self.width//2 - 5, self.y - 10, 10, 15))

        pygame.draw.polygon(screen, YELLOW, [
            (self.x + self.width//2 - 10, self.y + self.height),
            (self.x + self.width//2 + 10, self.y + self.height),
            (self.x + self.width//2, self.y + self.height + 10)
        ])

    def move(self, direction):
        if direction == "left" and self.x - self.speed > 0:
            self.x -= self.speed
        if direction == "right" and self.x + self.width + self.speed < WIDTH:
            self.x += self.speed

    def shoot(self):
        if self.cooldown <= 0:
            bullet = Bullet(self.x + self.width // 2, self.y)
            self.bullets.append(bullet)
            self.cooldown = self.cooldown_duration
            try:
                bullet_sound = mixer.Sound("bullet.wav")
                bullet_sound.set_volume(0.2)
                bullet_sound.play()
            except:
                pass

    def update(self):
        if self.cooldown > 0:
            self.cooldown -= 1

        for bullet in self.bullets[:]:
            bullet.update()
            if bullet.y < 0:
                self.bullets.remove(bullet)

    def draw_bullets(self):
        for bullet in self.bullets:
            bullet.draw()

    def get_hit_box(self):
        hitbox_width = self.width - 20
        hitbox_height = self.height - 10
        hitbox_x = self.x + 10
        hitbox_y = self.y + 5
        return pygame.Rect(hitbox_x, hitbox_y, hitbox_width, hitbox_height)

class Enemy:
    def __init__(self, x, y, enemy_type):
        self.width = 40
        self.height = 30
        self.x = x
        self.y = y
        self.speed = 1
        self.direction = 1
        self.enemy_type = enemy_type

        if enemy_type == 0:
            self.color = PURPLE
            self.points = 30
        elif enemy_type == 1:
            self.color = BLUE
            self.points = 20
        else:
            self.color = RED
            self.points = 10

    def draw(self):
        if self.enemy_type == 0:
            pygame.draw.ellipse(screen, self.color, (self.x, self.y + 10, self.width, self.height - 10))
            pygame.draw.ellipse(screen, WHITE, (self.x + 10, self.y, self.width - 20, 15))

        elif self.enemy_type == 1:
            pygame.draw.rect(screen, self.color, (self.x + 5, self.y + 5, self.width - 10, self.height - 5))
            pygame.draw.rect(screen, self.color, (self.x, self.y + 10, 5, self.height - 10))
            pygame.draw.rect(screen, self.color, (self.x + self.width - 5, self.y + 10, 5, self.height - 10))
            pygame.draw.circle(screen, WHITE, (self.x + 15, self.y + 12), 3)
            pygame.draw.circle(screen, WHITE, (self.x + self.width - 15, self.y + 12), 3)

        else:
            pygame.draw.rect(screen, self.color, (self.x + 5, self.y, self.width - 10, self.height))
            pygame.draw.rect(screen, self.color, (self.x, self.y + 5, 5, 10))
            pygame.draw.rect(screen, self.color, (self.x + self.width - 5, self.y + 5, 5, 10))
            pygame.draw.circle(screen, WHITE, (self.x + 15, self.y + 10), 3)
            pygame.draw.circle(screen, WHITE, (self.x + self.width - 15, self.y + 10), 3)

    def update(self):
        self.x += self.speed * self.direction

    def get_hit_box(self):
        hitbox_width = self.width - 10
        hitbox_height = self.height - 5
        hitbox_x = self.x + 5
        hitbox_y = self.y + 2
        return pygame.Rect(hitbox_x, hitbox_y, hitbox_width, hitbox_height)

class Bullet:
    def __init__(self, x, y):
        self.width = 6
        self.height = 15
        self.x = x - self.width // 2
        self.y = y
        self.speed = 12

    def draw(self):
        pygame.draw.rect(screen, YELLOW, (self.x, self.y, self.width, self.height))
        pygame.draw.rect(screen, WHITE, (self.x + 1, self.y, self.width - 2, self.height - 5))

    def update(self):
        self.y -= self.speed

    def get_hit_box(self):
        return pygame.Rect(self.x, self.y, self.width, self.height)

class EnemyBullet:
    def __init__(self, x, y):
        self.width = 4
        self.height = 12
        self.x = x
        self.y = y
        self.speed = 5

    def draw(self):
        pygame.draw.rect(screen, RED, (self.x, self.y, self.width, self.height))

    def update(self):
        self.y += self.speed

    def get_hit_box(self):
        return pygame.Rect(self.x, self.y, self.width, self.height)

class Explosion:
    def __init__(self, x, y):
        self.x = x
        self.y = y
        self.size = 20
        self.lifetime = 10
        self.current_frame = 0

    def draw(self):
        radius = int(self.size * (self.current_frame / self.lifetime))
        alpha = 255 - int(255 * (self.current_frame / self.lifetime))

        temp_surface = pygame.Surface((radius * 2, radius * 2), pygame.SRCALPHA)
        pygame.draw.circle(temp_surface, (255, 200, 0, alpha), (radius, radius), radius)

        screen.blit(temp_surface, (self.x - radius, self.y - radius))

    def update(self):
        self.current_frame += 1

class Barrier:
    def __init__(self, x, y):
        self.width = 80
        self.height = 50
        self.x = x
        self.y = y
        self.health = 4
        self.blocks = []

        block_size = 10
        for row in range(self.height // block_size):
            for col in range(self.width // block_size):
                if not (row > 2 and col > 1 and col < self.width // block_size - 2):
                    self.blocks.append({
                        'rect': pygame.Rect(self.x + col * block_size, self.y + row * block_size, block_size, block_size),
                        'health': self.health
                    })

    def draw(self):
        for block in self.blocks:
            if block['health'] == 4:
                color = GREEN
            elif block['health'] == 3:
                color = (150, 200, 0)
            elif block['health'] == 2:
                color = YELLOW
            else:
                color = RED

            pygame.draw.rect(screen, color, block['rect'])

    def check_collision(self, bullet_rect):
        for block in self.blocks[:]:
            if block['rect'].colliderect(bullet_rect):
                block['health'] -= 1
                if block['health'] <= 0:
                    self.blocks.remove(block)
                return True
        return False

class Game:
    def __init__(self):
        self.player = Player()
        self.enemies = []
        self.enemy_bullets = []
        self.explosions = []
        self.barriers = []
        self.create_enemies()
        self.create_barriers()
        self.score = 0
        self.game_over = False
        self.game_won = False
        self.level = 1
        self.enemy_shoot_cooldown = 0
        self.enemy_move_timer = 0
        self.enemy_move_delay = 30
        self.enemy_direction_changed = False

        try:
            mixer.init()
            mixer.music.load("background.wav")
            mixer.music.set_volume(0.3)
            mixer.music.play(-1)
        except:
            print("Could not load sounds. Continuing without audio.")

    def create_enemies(self):
        self.enemies.clear()
        rows = 5
        cols = 8

        x_spacing = 60
        y_spacing = 50

        start_x = (WIDTH - (cols - 1) * x_spacing) // 2
        start_y = 80

        for row in range(rows):
            for col in range(cols):
                x = start_x + col * x_spacing
                y = start_y + row * y_spacing
                if row == 0:
                    enemy_type = 0
                elif row < 3:
                    enemy_type = 1
                else:
                    enemy_type = 2

                self.enemies.append(Enemy(x, y, enemy_type))

    def create_barriers(self):
        self.barriers.clear()
        barrier_count = 4
        spacing = WIDTH // (barrier_count + 1)

        for i in range(barrier_count):
            x = spacing * (i + 1) - 40
            y = HEIGHT - 150
            self.barriers.append(Barrier(x, y))

    def check_collisions(self):

        for bullet in self.player.bullets[:]:
            bullet_rect = bullet.get_hit_box()


            barrier_hit = False
            for barrier in self.barriers:
                if barrier.check_collision(bullet_rect):
                    if bullet in self.player.bullets:
                        self.player.bullets.remove(bullet)
                    barrier_hit = True
                    break

            if barrier_hit:
                continue


            for enemy in self.enemies[:]:
                enemy_rect = enemy.get_hit_box()

                if bullet_rect.colliderect(enemy_rect):

                    self.explosions.append(Explosion(enemy.x + enemy.width // 2, enemy.y + enemy.height // 2))


                    if bullet in self.player.bullets:
                        self.player.bullets.remove(bullet)
                    self.enemies.remove(enemy)


                    self.score += enemy.points


                    try:
                        explosion_sound = mixer.Sound("explosion.wav")
                        explosion_sound.set_volume(0.2)
                        explosion_sound.play()
                    except:
                        pass

                    break


        player_rect = self.player.get_hit_box()
        for bullet in self.enemy_bullets[:]:
            bullet_rect = bullet.get_hit_box()


            barrier_hit = False
            for barrier in self.barriers:
                if barrier.check_collision(bullet_rect):
                    if bullet in self.enemy_bullets:
                        self.enemy_bullets.remove(bullet)
                    barrier_hit = True
                    break

            if barrier_hit:
                continue


            if bullet_rect.colliderect(player_rect):
                if bullet in self.enemy_bullets:
                    self.enemy_bullets.remove(bullet)
                self.player.lives -= 1


                try:
                    hit_sound = mixer.Sound("hit.wav")
                    hit_sound.set_volume(0.3)
                    hit_sound.play()
                except:
                    pass

                if self.player.lives <= 0:
                    self.game_over = True

    def enemy_shoot(self):
        if self.enemy_shoot_cooldown <= 0 and len(self.enemies) > 0:

            enemy = random.choice(self.enemies)
            bullet = EnemyBullet(enemy.x + enemy.width // 2, enemy.y + enemy.height)
            self.enemy_bullets.append(bullet)
            self.enemy_shoot_cooldown = random.randint(30, 100)

    def update_enemies(self):

        self.enemy_move_timer += 1
        if self.enemy_move_timer < self.enemy_move_delay:
            return

        self.enemy_move_timer = 0
        self.enemy_direction_changed = False


        for enemy in self.enemies:
            if ((enemy.x + enemy.width >= WIDTH - 10) and enemy.direction > 0) or (enemy.x <= 10 and enemy.direction < 0):
                self.enemy_direction_changed = True
                break


        for enemy in self.enemies:
            if self.enemy_direction_changed:
                enemy.direction *= -1
                enemy.y += 20
            enemy.update()


            if enemy.y + enemy.height >= HEIGHT - 80:
                self.game_over = True

    def update_enemy_bullets(self):

        for bullet in self.enemy_bullets[:]:
            bullet.update()
            if bullet.y > HEIGHT:
                self.enemy_bullets.remove(bullet)


        if self.enemy_shoot_cooldown > 0:
            self.enemy_shoot_cooldown -= 1


        if random.random() < 0.02 and not self.game_over and not self.game_won:
            self.enemy_shoot()

    def update_explosions(self):
        for explosion in self.explosions[:]:
            explosion.update()
            if explosion.current_frame >= explosion.lifetime:
                self.explosions.remove(explosion)

    def draw(self):

        screen.fill(BLACK)


        for _ in range(100):
            x = random.randint(0, WIDTH-1)
            y = random.randint(0, HEIGHT-1)
            size = random.randint(1, 2)
            brightness = random.randint(100, 255)
            pygame.draw.circle(screen, (brightness, brightness, brightness), (x, y), size)


        for barrier in self.barriers:
            barrier.draw()


        self.player.draw()
        self.player.draw_bullets()


        for enemy in self.enemies:
            enemy.draw()


        for bullet in self.enemy_bullets:
            bullet.draw()


        for explosion in self.explosions:
            explosion.draw()


        score_text = font.render(f"Score: {self.score}", True, WHITE)
        screen.blit(score_text, (10, 10))


        level_text = font.render(f"Level: {self.level}", True, WHITE)
        screen.blit(level_text, (WIDTH - 150, 10))


        lives_text = font.render(f"Lives: {self.player.lives}", True, WHITE)
        screen.blit(lives_text, (WIDTH // 2 - 50, 10))


        if self.game_over:

            overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
            overlay.fill((0, 0, 0, 128))
            screen.blit(overlay, (0, 0))

            game_over_text = font.render("GAME OVER", True, RED)
            screen.blit(game_over_text, (WIDTH//2 - game_over_text.get_width()//2, HEIGHT//2 - 50))

            restart_text = font.render("Press R to Restart", True, WHITE)
            screen.blit(restart_text, (WIDTH//2 - restart_text.get_width()//2, HEIGHT//2 + 10))


        if len(self.enemies) == 0 and not self.game_over:

            overlay = pygame.Surface((WIDTH, HEIGHT), pygame.SRCALPHA)
            overlay.fill((0, 0, 0, 128))
            screen.blit(overlay, (0, 0))

            level_complete_text = font.render(f"Level {self.level} Complete!", True, GREEN)
            screen.blit(level_complete_text, (WIDTH//2 - level_complete_text.get_width()//2, HEIGHT//2 - 50))

            next_level_text = font.render("Press N for Next Level", True, WHITE)
            screen.blit(next_level_text, (WIDTH//2 - next_level_text.get_width()//2, HEIGHT//2 + 10))


        controls_text = small_font.render("Controls: Arrows to move, Space to shoot, R to restart, N for next level", True, WHITE)
        screen.blit(controls_text, (10, HEIGHT - 30))

    def next_level(self):
        if len(self.enemies) == 0:
            self.level += 1
            self.create_enemies()
            self.create_barriers()

            for enemy in self.enemies:
                enemy.speed = 1 + (self.level * 0.2)

            self.enemy_move_delay = max(10, 30 - self.level * 2)

    def restart(self):
        self.__init__()
def main():
    game = Game()
    running = True

    while running:
        clock.tick(FPS)


        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                running = False

            if event.type == pygame.KEYDOWN:
                if event.key == pygame.K_r and game.game_over:
                    game.restart()
                if event.key == pygame.K_n and len(game.enemies) == 0:
                    game.next_level()
                if event.key == pygame.K_SPACE and not game.game_over and not game.game_won:
                    game.player.shoot()


        keys = pygame.key.get_pressed()
        if keys[pygame.K_LEFT]:
            game.player.move("left")
        if keys[pygame.K_RIGHT]:
            game.player.move("right")

        if keys[pygame.K_r] and keys[pygame.K_LCTRL]:
            game.restart()


        if not game.game_over and len(game.enemies) > 0:
            game.player.update()
            game.update_enemies()
            game.update_enemy_bullets()
            game.update_explosions()
            game.check_collisions()


        game.draw()


        pygame.display.flip()

    pygame.quit()
    sys.exit()

if __name__ == "__main__":
    main()